In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/rekhaamul/task-1/Task 1/CRYPTO-REDDIT-OPINION-TRAIN.csv
/kaggle/input/datasets/rekhaamul/task-1/Task 1/CRYPTO-TWITTER-OPINION-TRAIN.csv
/kaggle/input/datasets/rekhaamul/task-1/Task 1/CRYPTO-YOUTUBE-OPINION-TRAIN.csv
/kaggle/input/datasets/rekhaamul/crypto1/Task 2/mcq_train.jsonl
/kaggle/input/datasets/rekhaamul/crypto1/Task 2/QnA-TRAIN.csv
/kaggle/input/datasets/rekhaamul/crypto1/Task 2/qa_train.jsonl
/kaggle/input/datasets/rekhaamul/crypto1/Task 2/videos.jsonl
/kaggle/input/datasets/rekhaamul/crypto1/Task 2/qa_val.jsonl
/kaggle/input/datasets/rekhaamul/crypto1/Task 2/mcq_val.jsonl


# CryptoQA-X v3.0
## FIRE 2026 - Task 1: Cryptocurrency Opinion Classification

### Project Information
- Competition: FIRE 2026 CryptoQA
- Task: Task 1 - Opinion Classification
- Framework: Scikit-Learn + Hugging Face Transformers
- Version: 3.0

In [2]:
# ============================================================
# SECTION 2 - Import Libraries
# ============================================================

# Standard Libraries
import os
import warnings
import random

# Data Manipulation
import numpy as np
import pandas as pd

# Visualization

import matplotlib.pyplot as plt

# Machine Learning
from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.svm import LinearSVC

from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Save Models
import joblib

# Ignore warnings
warnings.filterwarnings("ignore")

print("✅ All libraries imported successfully.")

✅ All libraries imported successfully.


In [3]:
# ============================================================
# SECTION 3 - Reproducibility
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print(f"Random Seed Set: {SEED}")

Random Seed Set: 42


In [4]:
# ============================================================
# SECTION 4 - Create Project Folders
# ============================================================

folders = [
    "models",
    "outputs",
    "plots",
    "submissions",
    "checkpoints"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully.")

print("\nCreated folders:")

for folder in folders:
    print("-", folder)

Project folders created successfully.

Created folders:
- models
- outputs
- plots
- submissions
- checkpoints


In [5]:
# ============================================================
# SECTION 5 - Dataset Paths
# ============================================================

REDDIT_PATH = "/kaggle/input/datasets/rekhaamul/task-1/Task 1/CRYPTO-REDDIT-OPINION-TRAIN.csv"

TWITTER_PATH = "/kaggle/input/datasets/rekhaamul/task-1/Task 1/CRYPTO-TWITTER-OPINION-TRAIN.csv"

YOUTUBE_PATH = "/kaggle/input/datasets/rekhaamul/task-1/Task 1/CRYPTO-YOUTUBE-OPINION-TRAIN.csv"

print("✅ Dataset paths configured successfully.")

✅ Dataset paths configured successfully.


In [6]:
# ============================================================
# SECTION 6 - Load Datasets
# ============================================================
#import pandas as pd 

reddit = pd.read_csv(REDDIT_PATH)
twitter = pd.read_csv(TWITTER_PATH)
youtube = pd.read_csv(YOUTUBE_PATH)

print("Datasets Loaded Successfully!\n")

print("Reddit :", reddit.shape)
print("Twitter:", twitter.shape)
print("YouTube:", youtube.shape)

Datasets Loaded Successfully!

Reddit : (5000, 7)
Twitter: (4288, 4)
YouTube: (5000, 5)


# Section 7 - Data Standardization

The Reddit, Twitter, and YouTube datasets have different column names.
This section converts them into one common format so that all models can be trained on a single unified dataset.

In [7]:
# ============================================================
# SECTION 7 - Standardize Dataset Format
# ============================================================

# ---------- Reddit ----------
reddit_df = pd.DataFrame({
    "text": reddit["MAIN"].fillna(""),
    "platform": "reddit",
    "Level 1": reddit["Level 1"].astype(int),
    "Level 2": reddit["Level 2"],
    "Level 3": reddit["Level 3"]
})

# ---------- Twitter ----------
twitter_df = pd.DataFrame({
    "text": twitter["Tweet"].fillna(""),
    "platform": "twitter",
    "Level 1": twitter["Level 1"].astype(int),
    "Level 2": twitter["Level 2"],
    "Level 3": twitter["Level 3"]
})

# ---------- YouTube ----------
youtube.columns = youtube.columns.str.strip()

youtube_df = pd.DataFrame({
    "text": youtube["comment"].fillna(""),
    "platform": "youtube",
    "Level 1": youtube["Level 1"].astype(int),
    "Level 2": youtube["Level 2"],
    "Level 3": youtube["Level 3"]
})

print("✅ Dataset standardization completed.")

✅ Dataset standardization completed.


# Section 8 - Merge Datasets

Combine Reddit, Twitter, and YouTube into a single master dataset.

In [8]:
# ============================================================
# SECTION 8 - Merge Datasets
# ============================================================

master_df = pd.concat(
    [reddit_df, twitter_df, youtube_df],
    ignore_index=True
)

print("Master Dataset Shape:", master_df.shape)

master_df.head()

Master Dataset Shape: (14288, 5)


,text,platform,Level 1,Level 2,Level 3
0,Staking ADA,reddit,0,NaN,NaN
1,10 Most followed celebrities on Instagram,reddit,0,NaN,NaN
2,åŽäººç”¨æˆ·çŒæ°´,reddit,0,NaN,NaN
3,Your thoughts on SatoshiStreetBets,reddit,0,NaN,NaN
4,Happy New Year! By the time you;#8217;re readi...,reddit,0,NaN,NaN


In [9]:
# ============================================================
# SECTION 9 - Data Cleaning
# ============================================================

# Remove empty text
master_df = master_df[master_df["text"].str.strip() != ""]

# Remove duplicate text
master_df = master_df.drop_duplicates(subset="text")

# Reset index
master_df = master_df.reset_index(drop=True)

print("Dataset Shape After Cleaning:", master_df.shape)

Dataset Shape After Cleaning: (13771, 5)


# Section 10 - Dataset Overview

In [10]:
# ============================================================
# SECTION 10 - Dataset Overview
# ============================================================

print(master_df.info())

print("\n")

print(master_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13771 entries, 0 to 13770
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   text      13771 non-null  object 
 1   platform  13771 non-null  object 
 2   Level 1   13771 non-null  int64  
 3   Level 2   8973 non-null   float64
 4   Level 3   6250 non-null   float64
dtypes: float64(2), int64(1), object(2)
memory usage: 538.1+ KB
None


                                                text platform  Level 1  \
0                                       Staking ADA    reddit        0   
1          10 Most followed celebrities on Instagram   reddit        0   
2                                åŽäººç”¨æˆ·çŒæ°´    reddit        0   
3                Your thoughts on SatoshiStreetBets    reddit        0   
4  Happy New Year! By the time you;#8217;re readi...   reddit        0   

   Level 2  Level 3  
0      NaN      NaN  
1      NaN      NaN  
2      NaN      NaN  
3      NaN     

In [11]:
# ============================================================
# SECTION 11 - Class Distribution
# ============================================================

print(master_df["Level 1"].value_counts())

print("\n")

print(master_df["Level 1"].value_counts(normalize=True))

Level 1
2    8971
0    2703
1    2097
Name: count, dtype: int64


Level 1
2    0.651441
0    0.196282
1    0.152277
Name: proportion, dtype: float64


In [12]:
# ============================================================
# SECTION 12 - Train / Validation Split
# ============================================================

X = master_df["text"]

y = master_df["Level 1"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print("Training Samples :", len(X_train))
print("Validation Samples:", len(X_valid))

Training Samples : 11016
Validation Samples: 2755


In [13]:
# ============================================================
# SECTION 13 - TF-IDF
# ============================================================

vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    lowercase=True,
    sublinear_tf=True,
    min_df=2
)

X_train_tfidf = vectorizer.fit_transform(X_train)

X_valid_tfidf = vectorizer.transform(X_valid)

print("Training Matrix :", X_train_tfidf.shape)
print("Validation Matrix:", X_valid_tfidf.shape)

Training Matrix : (11016, 50000)
Validation Matrix: (2755, 50000)


In [14]:
# ============================================================
# SECTION 14 - Logistic Regression
# ============================================================

log_reg = LogisticRegression(
    max_iter=1000,
    random_state=SEED
)

log_reg.fit(X_train_tfidf, y_train)

log_pred = log_reg.predict(X_valid_tfidf)

log_acc = accuracy_score(y_valid, log_pred)

log_f1 = f1_score(
    y_valid,
    log_pred,
    average="macro"
)

print("Accuracy :", round(log_acc,4))
print("Macro F1 :", round(log_f1,4))

Accuracy : 0.8054
Macro F1 : 0.7126


In [15]:
# ============================================================
# SECTION 15 - Linear SVM
# ============================================================

svm = LinearSVC(random_state=SEED)

svm.fit(
    X_train_tfidf,
    y_train
)

svm_pred = svm.predict(X_valid_tfidf)

svm_acc = accuracy_score(
    y_valid,
    svm_pred
)

svm_f1 = f1_score(
    y_valid,
    svm_pred,
    average="macro"
)

print("Accuracy :", round(svm_acc,4))
print("Macro F1 :", round(svm_f1,4))

Accuracy : 0.8174
Macro F1 : 0.7469


In [16]:
# ============================================================
# SECTION 16 - Naive Bayes
# ============================================================

nb = MultinomialNB()

nb.fit(
    X_train_tfidf,
    y_train
)

nb_pred = nb.predict(
    X_valid_tfidf
)

nb_acc = accuracy_score(
    y_valid,
    nb_pred
)

nb_f1 = f1_score(
    y_valid,
    nb_pred,
    average="macro"
)

print("Accuracy :", round(nb_acc,4))
print("Macro F1 :", round(nb_f1,4))

Accuracy : 0.7368
Macro F1 : 0.5422


In [17]:
# ============================================================
# SECTION 17 - Model Comparison
# ============================================================

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Linear SVM",
        "Naive Bayes"
    ],
    "Accuracy": [
        log_acc,
        svm_acc,
        nb_acc
    ],
    "Macro F1": [
        log_f1,
        svm_f1,
        nb_f1
    ]
})

results = results.sort_values(
    by="Macro F1",
    ascending=False
)

results

,Model,Accuracy,Macro F1
1,Linear SVM,0.817423,0.746862
0,Logistic Regression,0.805445,0.712565
2,Naive Bayes,0.736842,0.542176


In [18]:
# ============================================================
# SECTION 18 - Save Models
# ============================================================

import joblib

# Save TF-IDF Vectorizer
joblib.dump(vectorizer, "models/tfidf_vectorizer.pkl")

# Save Logistic Regression
joblib.dump(log_reg, "models/logistic_regression.pkl")

# Save Linear SVM
joblib.dump(svm, "models/linear_svm.pkl")

# Save Naive Bayes
joblib.dump(nb, "models/naive_bayes.pkl")

print("✅ All baseline models saved successfully.")

✅ All baseline models saved successfully.


In [19]:
# ============================================================
# SECTION 19 - Save Results
# ============================================================

results.to_csv(
    "outputs/model_comparison.csv",
    index=False
)

print("✅ Model comparison saved.")

✅ Model comparison saved.


In [20]:
# ============================================================
# SECTION 20 - Save Predictions
# ============================================================

prediction_df = pd.DataFrame({
    "Text": X_valid,
    "Actual": y_valid,
    "Prediction": log_pred
})

prediction_df.to_csv(
    "outputs/logistic_predictions.csv",
    index=False
)

print("✅ Validation predictions saved.")

✅ Validation predictions saved.


In [21]:
# ============================================================
# SECTION 21 - Baseline Summary
# ============================================================

print("="*60)
print("CryptoQA-X Baseline Completed")
print("="*60)

print()

print(results)

print()

print("Best Model :",
      results.iloc[0]["Model"])

print("Best Macro F1 :",
      round(results.iloc[0]["Macro F1"],4))

print()

print("Project Status : BASELINE COMPLETE")

CryptoQA-X Baseline Completed

                 Model  Accuracy  Macro F1
1           Linear SVM  0.817423  0.746862
0  Logistic Regression  0.805445  0.712565
2          Naive Bayes  0.736842  0.542176

Best Model : Linear SVM
Best Macro F1 : 0.7469

Project Status : BASELINE COMPLETE


In [22]:
# ============================================================
# SECTION 22 - Verify Saved Files
# ============================================================

print("Models Folder")
print(os.listdir("models"))

print()

print("Outputs Folder")
print(os.listdir("outputs"))

Models Folder
['naive_bayes.pkl', 'tfidf_vectorizer.pkl', 'linear_svm.pkl', 'logistic_regression.pkl']

Outputs Folder
['logistic_predictions.csv', 'model_comparison.csv']


In [23]:
# ============================================================
# SECTION 23 - Check GPU
# ============================================================

import torch

print("PyTorch Version :", torch.__version__)

print("GPU Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name :", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch Version : 2.10.0+cu128
GPU Available : True
GPU Name : Tesla T4


In [24]:
# ============================================================
# SECTION 24 - Install Libraries
# ============================================================

!pip -q install transformers datasets accelerate evaluate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00


In [25]:
# ============================================================
# SECTION 25 - Import Libraries
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from datasets import Dataset

import evaluate

print("✅ Transformer libraries loaded.")

✅ Transformer libraries loaded.


In [26]:
# ============================================================
# SECTION 26 - Load DeBERTa
# ============================================================

MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer Loaded Successfully")

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Tokenizer Loaded Successfully


In [27]:
print(master_df.head())

print()

print(master_df.columns)

print()

print(master_df.shape)

                                                text platform  Level 1  \
0                                       Staking ADA    reddit        0   
1          10 Most followed celebrities on Instagram   reddit        0   
2                                åŽäººç”¨æˆ·çŒæ°´    reddit        0   
3                Your thoughts on SatoshiStreetBets    reddit        0   
4  Happy New Year! By the time you;#8217;re readi...   reddit        0   

   Level 2  Level 3  
0      NaN      NaN  
1      NaN      NaN  
2      NaN      NaN  
3      NaN      NaN  
4      NaN      NaN  

Index(['text', 'platform', 'Level 1', 'Level 2', 'Level 3'], dtype='object')

(13771, 5)


In [28]:
# ============================================================
# SECTION 27 - Prepare Dataset
# ============================================================

from sklearn.model_selection import train_test_split

# Keep only required columns
df = master_df[["text", "Level 1"]].copy()

# Rename label column for Hugging Face
df.rename(columns={"Level 1": "label"}, inplace=True)

# Convert labels to integer
df["label"] = df["label"].astype(int)

# Train-validation split
train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Training Samples :", len(train_df))
print("Validation Samples :", len(valid_df))

Training Samples : 11016
Validation Samples : 2755


In [29]:
# ============================================================
# SECTION 28 - Hugging Face Dataset
# ============================================================

from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)

print(train_dataset)
print(valid_dataset)

Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 11016
})
Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 2755
})


# section 28(new version)

In [30]:
# ============================================================
# STEP 1: Create Datasets & Tokenize Text
# ============================================================

from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 1. Convert pandas DataFrames to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)

# 2. Define Tokenization Function
def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        truncation=True, 
        max_length=128,          # Adjust max_length if your project used a specific length
        padding="max_length"
    )

# 3. Apply Tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)

print("✅ Datasets created and tokenized successfully!")
print("Sample Train Features:", train_dataset.column_names)

Map:   0%|          | 0/11016 [00:00<?, ? examples/s]

Map:   0%|          | 0/2755 [00:00<?, ? examples/s]

✅ Datasets created and tokenized successfully!
Sample Train Features: ['text', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask']


In [31]:
# ============================================================
# SECTION 29 - Tokenization
# ============================================================

MAX_LENGTH = 128

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
valid_dataset = valid_dataset.map(tokenize_function, batched=True)

print("Tokenization Completed Successfully")

Map:   0%|          | 0/11016 [00:00<?, ? examples/s]

Map:   0%|          | 0/2755 [00:00<?, ? examples/s]

Tokenization Completed Successfully


In [ ]:
# ============================================================
# TRAIN, EVALUATE & SAVE MODEL
# ============================================================

import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    EarlyStoppingCallback
)

MODEL_NAME = "microsoft/deberta-v3-base"

# 1. Setup Metrics & Class Weights
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'macro_f1': f1, 'precision': precision, 'recall': recall}

raw_labels = train_dataset["label"]
if isinstance(raw_labels, torch.Tensor):
    train_labels = raw_labels.cpu().numpy().astype(int)
else:
    train_labels = np.array(raw_labels, dtype=int)

classes = np.unique(train_labels)
computed_weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_labels)
CLASS_WEIGHTS = torch.tensor(computed_weights, dtype=torch.float32)

# 2. Model & Custom Weighted Trainer Setup (FP32 Clean Run)
torch.cuda.empty_cache()
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, ignore_mismatched_sizes=True
).to(torch.float32)

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        if self.class_weights is not None:
            weights = self.class_weights.to(device=logits.device, dtype=torch.float32)
            loss_fct = nn.CrossEntropyLoss(weight=weights)
        else:
            loss_fct = nn.CrossEntropyLoss()
            
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir="./deberta_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_steps=100,
    max_grad_norm=1.0,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=False,
    bf16=False,
    report_to="none"
)

trainer = WeightedTrainer(
    class_weights=CLASS_WEIGHTS,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("🚀 Starting Training Loop...")
trainer.train()

# 3. Evaluate & Save Final Model
print("\n📊 Evaluating Best Model...")
final_metrics = trainer.evaluate()
print("Final Validation Metrics:", final_metrics)

SAVE_PATH = "./deberta_finetuned_model"
trainer.save_model(SAVE_PATH)
print(f"💾 Model saved successfully to '{SAVE_PATH}'!")

In [ ]:
# ============================================================
# SECTION 30 - Dataset Formatting
# ============================================================

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

valid_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

print(train_dataset[0])

In [ ]:
# ============================================================
# SECTION 31 - Load Model
# ============================================================

import torch

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

print("Model Loaded Successfully")

In [ ]:
# ============================================================
# SECTION 32 - Evaluation Metrics
# ============================================================

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

print("Evaluation Libraries Loaded Successfully")

In [ ]:
# ============================================================
# SECTION 33 - Metrics Function
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "macro_f1": f1,
        "precision": precision,
        "recall": recall
    }

print("Metrics Function Created")

In [ ]:
# ============================================================
# SECTION 34 - Training Arguments
# ============================================================

from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="./deberta_output",

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_strategy="steps",

    logging_steps=100,

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",

    greater_is_better=True,

    fp16=False,

    report_to="none",

    seed=42
)

print("Training Arguments Ready")

In [ ]:
# ============================================================
# SECTION 35 - Create Trainer
# ============================================================

from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

print("✅ Trainer Created Successfully")

# FIX code

In [ ]:
# ============================================================
# FIX STEP 1: Re-initialize DeBERTa-v3 Model
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "microsoft/deberta-v3-base"  # Or your specific DeBERTa model path

# 1. Clear GPU Cache to free memory from the failed run
torch.cuda.empty_cache()

# 2. Reload clean model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    ignore_mismatched_sizes=True
)

print(" Clean DeBERTa-v3 model re-loaded into memory.")

In [ ]:
# ============================================================
# STEP 1: Force Clean Float32 DeBERTa Model
# ============================================================

import torch
import shutil
import os
from transformers import AutoModelForSequenceClassification, TrainingArguments

# 1. Clear GPU memory cache
torch.cuda.empty_cache()

# 2. Delete existing broken checkpoints directory if it exists
if os.path.exists("./deberta_checkpoints"):
    shutil.rmtree("./deberta_checkpoints")
    print("🧹 Cleared old corrupted checkpoints directory.")

MODEL_NAME = "microsoft/deberta-v3-base"

# 3. Explicitly load model in float32
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    ignore_mismatched_sizes=True
).to(torch.float32)

# 4. Strictly force FP32 Training Arguments
training_args = TrainingArguments(
    output_dir="./deberta_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_steps=100,
    max_grad_norm=1.0,                    # Clips extreme gradients
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=False,                           # Forced OFF to prevent NaN loss
    bf16=False,                           # Forced OFF
    report_to="none"
)

print("✅ Fresh float32 model & safe arguments loaded successfully!")

In [ ]:
# ============================================================
# STEP A: Clean Model & Safe Training Arguments
# ============================================================

import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments

MODEL_NAME = "microsoft/deberta-v3-base"  # Adjust if using a different DeBERTa variant

# 1. Clear GPU Memory
torch.cuda.empty_cache()

# 2. Reload a fresh, clean copy of DeBERTa
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    ignore_mismatched_sizes=True
)

# 3. Safe Training Arguments (fp16 MUST be False for DeBERTa-v3)
training_args = TrainingArguments(
    output_dir="./deberta_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_steps=100,
    max_grad_norm=1.0,                    # Prevents gradient explosions
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=False,                           # Prevents NaN loss overflow!
    bf16=False,                           
    report_to="none"
)

print(" Clean model loaded & safe TrainingArguments set!")

In [ ]:
# ============================================================
# STEP 2: Execute Training Loop
# ============================================================

import torch.nn as nn
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from transformers import Trainer, EarlyStoppingCallback

# 1. Extract labels & calculate class weights
raw_labels = train_dataset["label"]
if isinstance(raw_labels, torch.Tensor):
    train_labels = raw_labels.cpu().numpy().astype(int)
else:
    train_labels = np.array(raw_labels, dtype=int)

classes = np.unique(train_labels)
computed_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_labels
)
CLASS_WEIGHTS = torch.tensor(computed_weights, dtype=torch.float32)


# 2. Define Custom Trainer forcing Float32 Loss
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        if self.class_weights is not None:
            weights = self.class_weights.to(device=logits.device, dtype=torch.float32)
            loss_fct = nn.CrossEntropyLoss(weight=weights)
        else:
            loss_fct = nn.CrossEntropyLoss()
            
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


# 3. Launch Training
trainer = WeightedTrainer(
    class_weights=CLASS_WEIGHTS,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("🚀 Starting Clean FP32 Training...")
trainer.train()

In [ ]:
import joblib

joblib.dump(svm, "best_svm_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("Saved Successfully")

In [ ]:
import os

print(os.listdir())

In [ ]:
def predict_text(text):
    x = vectorizer.transform([text])
    pred = svm.predict(x)[0]
    return pred

In [ ]:
print(predict_text("Bitcoin price is rising rapidly"))
print(predict_text("Ethereum is a scam"))

In [ ]:
# ============================================================
# TASK 2 - Load Datasets
# ============================================================

import pandas as pd

qna_train = pd.read_csv(
    "/kaggle/input/datasets/rekhaamul/crypto1/Task 2/QnA-TRAIN.csv",
    encoding="latin1"
)
qa_val = pd.read_json(
    "/kaggle/input/datasets/rekhaamul/crypto1/Task 2/qa_val.jsonl",
    lines=True
)

mcq_train = pd.read_json(
    "/kaggle/input/datasets/rekhaamul/crypto1/Task 2/mcq_train.jsonl",
    lines=True
)

mcq_val = pd.read_json(
    "/kaggle/input/datasets/rekhaamul/crypto1/Task 2/mcq_val.jsonl",
    lines=True
)

videos = pd.read_json(
    "/kaggle/input/datasets/rekhaamul/crypto1/Task 2/videos.jsonl",
    lines=True
)

qna_train = pd.read_csv(
    "/kaggle/input/datasets/rekhaamul/crypto1/Task 2/QnA-TRAIN.csv"
)

print("All Task 2 datasets loaded successfully!")

In [ ]:
import pandas as pd

qna_train = pd.read_csv(
    "/kaggle/input/datasets/rekhaamul/crypto1/Task 2/QnA-TRAIN.csv",
    encoding="ISO-8859-1",
    engine="python",
    on_bad_lines="skip"
)

print(qna_train.head())
print(qna_train.shape)

In [ ]:
def generate_answer(question):

    row = retrieve_chunk(question)

    prompt = build_prompt(question, row["chunk_text"])

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.2,
        do_sample=False
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [ ]:
sample_question = qa_train.iloc[0]["question"]

print("QUESTION:")
print(sample_question)

print()

print("EXPECTED:")
print(qa_train.iloc[0]["answer"])

print()

print("GENERATED:")
print(generate_answer(sample_question))

In [ ]:
from sklearn.metrics.pairwise import linear_kernel
import numpy as np

def retrieve_chunks(question, top_k=3):

    question_vector = retriever.transform([question])

    scores = linear_kernel(question_vector, chunk_vectors).flatten()

    best_indices = np.argsort(scores)[-top_k:][::-1]

    return chunk_df.iloc[best_indices]

# re-running SECTION-30

In [ ]:
# ============================================================
# STEP 0: Ensure train_dataset is loaded
# ============================================================

# Re-apply dataset formatting (Section 30)
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

valid_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

print("✅ train_dataset is ready in memory!")

# improving the model's accuracy

In [ ]:
# ============================================================
# CREATE DATASETS & TOKENIZE (MAX_LENGTH = 256)
# ============================================================

from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 1. Re-build Hugging Face Datasets from DataFrames
train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)

# 2. Tokenize function with max_length=256
def tokenize_function_256(examples):
    return tokenizer(
        examples["text"], 
        truncation=True, 
        max_length=256, 
        padding="max_length"
    )

print("🔄 Re-tokenizing datasets with max_length=256...")
train_dataset = train_dataset.map(tokenize_function_256, batched=True)
valid_dataset = valid_dataset.map(tokenize_function_256, batched=True)
print("✅ Tokenization complete!")

In [ ]:
import os
import gc
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.optimize import minimize
from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer, 
    EarlyStoppingCallback
)

# 1. RESOLVE DATA & CLEAN DISK
for item in os.listdir("."):
    if item.startswith("deberta_fold_") or item.startswith("best_model_fold_"):
        shutil.rmtree(item, ignore_errors=True)

if 'master_df' not in locals():
    if 'train_df' in locals():
        master_df = train_df.copy()
    elif 'df' in locals():
        master_df = df.copy()

text_col = 'text' if 'text' in master_df.columns else master_df.columns[0]
master_df[text_col] = master_df[text_col].fillna("").astype(str)

if "label" not in master_df.columns and "Level 1" in master_df.columns:
    master_df["label"] = master_df["Level 1"].astype(int)

# 2. CONFIGURATION
MODEL_NAME = "microsoft/deberta-v3-large"
MAX_LEN = 256
NUM_LABELS = len(master_df["label"].unique())
N_SPLITS = 3
SEED = 42

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
master_df["fold"] = -1
for fold, (_, val_idx) in enumerate(skf.split(master_df, master_df["label"])):
    master_df.loc[val_idx, "fold"] = fold

# 3. CALCULATE CLASS WEIGHTS
class_weights = compute_class_weight(
    class_weight="balanced", 
    classes=np.unique(master_df["label"]), 
    y=master_df["label"].values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).cuda()

# FIXED WEIGHTED TRAINER WITH HALF/BF16 DTYPE MATCHING
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # FIX: Ensure class_weights_tensor matches the exact dtype of logits (BF16/FP32)
        weights = class_weights_tensor.to(dtype=logits.dtype, device=logits.device)
        
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'macro_f1': f1, 'precision': precision, 'recall': recall}

def tokenize_fn(examples):
    return tokenizer(examples[text_col], truncation=True, max_length=MAX_LEN, padding="max_length")

# 4. TRAINING LOOP
print(f"🚀 Training Native Pipeline with {MODEL_NAME}...")

fold_models = []
oof_predictions = np.zeros((len(master_df), NUM_LABELS))

for fold in range(N_SPLITS):
    print(f"\n==================== TRAINING FOLD {fold + 1} / {N_SPLITS} ====================")
    
    train_fold_df = master_df[master_df["fold"] != fold].reset_index(drop=True)
    val_fold_df = master_df[master_df["fold"] == fold].reset_index(drop=True)
    val_indices = master_df[master_df["fold"] == fold].index
    
    train_ds = Dataset.from_pandas(train_fold_df).map(tokenize_fn, batched=True)
    val_ds = Dataset.from_pandas(val_fold_df).map(tokenize_fn, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

    fold_output_dir = f"./deberta_fold_{fold}"

    training_args = TrainingArguments(
        output_dir=fold_output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        save_only_model=True,
        learning_rate=3e-5,               # Higher LR for fast convergence
        lr_scheduler_type="cosine",
        warmup_steps=100,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,
        num_train_epochs=4,               
        weight_decay=0.01,
        max_grad_norm=1.0,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=False,
        bf16=USE_BF16,
        report_to="none"
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()
    
    val_preds = trainer.predict(val_ds)
    val_probs = torch.nn.functional.softmax(torch.tensor(val_preds.predictions), dim=-1).numpy()
    oof_predictions[val_indices] = val_probs

    save_fold_path = f"./best_model_fold_{fold}"
    model.save_pretrained(save_fold_path)
    tokenizer.save_pretrained(save_fold_path)
    fold_models.append(save_fold_path)
    
    shutil.rmtree(fold_output_dir, ignore_errors=True)
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

# 5. THRESHOLD OPTIMIZATION
def f1_loss(weights, probs, targets):
    scaled_probs = probs * weights
    preds = np.argmax(scaled_probs, axis=-1)
    _, _, f1, _ = precision_recall_fscore_support(targets, preds, average='macro', zero_division=0)
    return -f1

opt_res = minimize(f1_loss, [1.0] * NUM_LABELS, args=(oof_predictions, master_df["label"].values), method='Nelder-Mead')
optimal_weights = opt_res.x

oof_preds_opt = np.argmax(oof_predictions * optimal_weights, axis=-1)
opt_acc = accuracy_score(master_df["label"], oof_preds_opt)
opt_prec, opt_rec, opt_f1, _ = precision_recall_fscore_support(master_df["label"], oof_preds_opt, average='macro', zero_division=0)

print("\n================ FINAL METRICS ================")
print(f"Optimized Accuracy : {opt_acc:.4f}")
print(f"Optimized Macro F1 : {opt_f1:.4f}")
print("===============================================")

# CryptoQA Task-2 

In [32]:
import os
import glob

# ------------------------------------------------------------
# DYNAMIC KAGGLE PATH RESOLUTION FOR TASK 2
# ------------------------------------------------------------
def find_task2_directory():
    base_dir = "/kaggle/input"
    if not os.path.exists(base_dir):
        base_dir = "." # Local fallback
        
    # Look specifically for files inside Task 2 dataset
    target_file = "mcq_train.jsonl"
    matches = glob.glob(f"{base_dir}/**/{target_file}", recursive=True)
    
    if matches:
        resolved_path = os.path.dirname(matches[0])
        print(f"✅ Auto-detected Task 2 path: '{resolved_path}'")
        return resolved_path
    
    # Fallback search for directories containing 'Task 2' or 'Task-2'
    for root, dirs, _ in os.walk(base_dir):
        for d in dirs:
            if "task" in d.lower() and "2" in d:
                resolved_path = os.path.join(root, d)
                print(f"✅ Found matching folder: '{resolved_path}'")
                return resolved_path

    raise FileNotFoundError("❌ Could not automatically locate Task 2 files under /kaggle/input/.")

TASK2_DIR = find_task2_directory()
print(f"Available files in directory: {os.listdir(TASK2_DIR)}")

✅ Auto-detected Task 2 path: '/kaggle/input/datasets/rekhaamul/crypto1/Task 2'
Available files in directory: ['mcq_train.jsonl', 'QnA-TRAIN.csv', 'qa_train.jsonl', 'videos.jsonl', 'qa_val.jsonl', 'mcq_val.jsonl']


In [33]:
import json
import pandas as pd

# Load one entry from mcq_train.jsonl
with open('/kaggle/input/datasets/rekhaamul/crypto1/Task 2/mcq_train.jsonl', 'r', encoding='utf-8', errors='ignore') as f:
    sample_mcq = json.loads(f.readline())

# Load QnA-TRAIN.csv header with latin1 encoding to prevent byte error
qna_df = pd.read_csv('/kaggle/input/datasets/rekhaamul/crypto1/Task 2/QnA-TRAIN.csv', nrows=2, encoding='latin1')

print("--- MCQ JSONL Sample Keys ---")
print(list(sample_mcq.keys()))
print("Sample Content:", sample_mcq)

print("\n--- QnA CSV Columns ---")
print(list(qna_df.columns))

--- MCQ JSONL Sample Keys ---
['id', 'question', 'options', 'answer_index', 'video_id']
Sample Content: {'id': 'mcq-train-0001', 'question': "Given Bitcoin's current price of $73,000, what has been the evolution of its use case since 2015?", 'options': ["Bitcoin's use case has evolved from a transactional medium due to scalability issues to a store of value.", "Bitcoin's use case has evolved from a transactional medium due to cost constraints to a store of value.", "Ethereum's use case has evolved from a transactional medium due to cost constraints to a store of value.", "Bitcoin's use case has evolved from a transactional medium due to cost constraints to a transactional medium."], 'answer_index': 1, 'video_id': '-0Lysi1bz7Q'}

--- QnA CSV Columns ---
['title', 'selftext', 'MAIN', 'comment_body', 'relevance']


# CrpytoBERT using code(1)

In [1]:
# ============================================================
# CRYPTOBERT HIGH-ACCURACY MCQ PIPELINE
# ============================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import gc
import json
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from dataclasses import dataclass
from typing import Optional, Union
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer, 
    EarlyStoppingCallback,
    PreTrainedTokenizerBase
)
from transformers.tokenization_utils_base import PaddingStrategy

# ------------------------------------------------------------
# 1. PATHS & DATA PARSING
# ------------------------------------------------------------
MCQ_TRAIN_PATH = "/kaggle/input/datasets/rekhaamul/crypto1/Task 2/mcq_train.jsonl"
MCQ_VAL_PATH   = "/kaggle/input/datasets/rekhaamul/crypto1/Task 2/mcq_val.jsonl"

def parse_mcq_jsonl(filepath):
    records = []
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            data = json.loads(line)
            question = data.get('question', '')
            options = data.get('options', [])
            
            while len(options) < 4:
                options.append("")
            options = options[:4]
            
            records.append({
                'id': data.get('id'),
                'prompt': question,
                'A': options[0],
                'B': options[1],
                'C': options[2],
                'D': options[3],
                'label': data.get('answer_index', 0)
            })
    return pd.DataFrame(records)

df = parse_mcq_jsonl(MCQ_TRAIN_PATH)
df = df.dropna(subset=['label']).reset_index(drop=True)
df['label'] = df['label'].astype(int)

train_df, val_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['label'])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"✅ Loaded {len(train_df)} training samples and {len(val_df)} validation samples.")

# ------------------------------------------------------------
# 2. TOKENIZATION & DATA COLLATOR
# ------------------------------------------------------------
MODEL_NAME = "ElKulako/cryptobert"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
option_cols = ['A', 'B', 'C', 'D']

def preprocess_mcq(examples):
    first_sentences = [[f"Question: {prompt}"] * 4 for prompt in examples['prompt']]
    second_sentences = [[f"Option: {examples[col][i]}" for col in option_cols] for i in range(len(examples['prompt']))]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized_examples = tokenizer(
        first_sentences, 
        second_sentences, 
        truncation=True, 
        max_length=MAX_LEN
    )
    
    return {
        k: [v[i : i + 4] for i in range(0, len(v), 4)]
        for k, v in tokenized_examples.items()
    }

@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0] else "labels"
        labels = [feature.pop(label_name) for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)]
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

# Custom Wrapper Adapter for Sequence Models on MCQ tasks
class CryptoBertForMultipleChoice(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1, ignore_mismatched_sizes=True)

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        num_choices = input_ids.shape[1]
        
        flat_input_ids = input_ids.view(-1, input_ids.size(-1))
        flat_attention_mask = attention_mask.view(-1, attention_mask.size(-1)) if attention_mask is not None else None

        outputs = self.encoder(input_ids=flat_input_ids, attention_mask=flat_attention_mask)
        logits = outputs.logits.view(-1, num_choices)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)

        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'macro_f1': f1, 'precision': precision, 'recall': recall}

train_ds = Dataset.from_pandas(train_df).map(preprocess_mcq, batched=True)
val_ds = Dataset.from_pandas(val_df).map(preprocess_mcq, batched=True)

# ------------------------------------------------------------
# 3. HIGH-ACCURACY STABLE TRAINING
# ------------------------------------------------------------
model = CryptoBertForMultipleChoice(MODEL_NAME)
output_dir = "./cryptobert_mcq_optimal"

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    learning_rate=2e-5,               # Optimized stable learning rate
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    per_device_train_batch_size=4,    
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,    # Effective Batch Size = 16
    num_train_epochs=6,
    weight_decay=0.01,
    max_grad_norm=1.0,                # Prevents NaN gradient explosion
    logging_steps=10,                 
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=False,                       # FP32 ensures strictly numerical stability
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("\n🚀 Starting Training with CryptoBERT...")
trainer.train()

# Evaluate Validation Metrics
eval_res = trainer.evaluate()
print("\n================ VALIDATION METRICS ================")
print(f"Validation Loss     : {eval_res['eval_loss']:.4f}")
print(f"Validation Accuracy : {eval_res['eval_accuracy']:.4f}")
print(f"Validation Macro F1 : {eval_res['eval_macro_f1']:.4f}")
print("====================================================")

# Save Model
model.encoder.save_pretrained("./best_cryptobert")
tokenizer.save_pretrained("./best_cryptobert")

# ------------------------------------------------------------
# 4. SUBMISSION GENERATION
# ------------------------------------------------------------
if os.path.exists(MCQ_VAL_PATH):
    test_df = parse_mcq_jsonl(MCQ_VAL_PATH)
    test_ds = Dataset.from_pandas(test_df).map(preprocess_mcq, batched=True)

    preds = trainer.predict(test_ds)
    final_test_preds = np.argmax(preds.predictions, axis=-1)

    submission_df = pd.DataFrame({
        'id': test_df['id'],
        'answer_index': final_test_preds
    })

    SUBMISSION_FILE = "sub_task2_fire2026.csv"
    submission_df.to_csv(SUBMISSION_FILE, index=False)
    print(f"\n🎉 SUBMISSION FILE SAVED SUCCESSFULLY: '{SUBMISSION_FILE}'")

✅ Loaded 850 training samples and 150 validation samples.


config.json:   0%|          | 0.00/932 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (8 outputs left out)


================ VALIDATION METRICS ================
Validation Loss     : 1.1612
Validation Accuracy : 0.5933
Validation Macro F1 : 0.5880


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]


🎉 SUBMISSION FILE SAVED SUCCESSFULLY: 'sub_task2_fire2026.csv'


# Finding file path

In [10]:
import os

# Base directory for Kaggle input data
base_dir = "/kaggle/input"

for root, dirs, files in os.walk(base_dir):
    for file in files:
        # Construct and print absolute path for each file
        abs_path = os.path.abspath(os.path.join(root, file))
        print(abs_path)

/kaggle/input/datasets/rekhaamul/crypto1/Task 2/mcq_train.jsonl
/kaggle/input/datasets/rekhaamul/crypto1/Task 2/QnA-TRAIN.csv
/kaggle/input/datasets/rekhaamul/crypto1/Task 2/qa_train.jsonl
/kaggle/input/datasets/rekhaamul/crypto1/Task 2/videos.jsonl
/kaggle/input/datasets/rekhaamul/crypto1/Task 2/qa_val.jsonl
/kaggle/input/datasets/rekhaamul/crypto1/Task 2/mcq_val.jsonl
/kaggle/input/datasets/rekhaamul/task-1/Task 1/CRYPTO-REDDIT-OPINION-TRAIN.csv
/kaggle/input/datasets/rekhaamul/task-1/Task 1/CRYPTO-TWITTER-OPINION-TRAIN.csv
/kaggle/input/datasets/rekhaamul/task-1/Task 1/CRYPTO-YOUTUBE-OPINION-TRAIN.csv


# CrpytoBERT using code(2)

In [ ]:
# ============================================================
# HIGH-ACCURACY MCQ PIPELINE (ROBERTA-LARGE TARGET > 0.80)
# ============================================================

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import gc
import json
import numpy as np
import pandas as pd
import torch
from dataclasses import dataclass
from typing import Optional, Union
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments, 
    Trainer, 
    EarlyStoppingCallback,
    PreTrainedTokenizerBase
)
from transformers.tokenization_utils_base import PaddingStrategy

# ------------------------------------------------------------
# 1. DATA PREPARATION
# ------------------------------------------------------------
MCQ_TRAIN_PATH = "/kaggle/input/datasets/rekhaamul/crypto1/Task 2/mcq_train.jsonl"
MCQ_VAL_PATH   = "/kaggle/input/datasets/rekhaamul/crypto1/Task 2/mcq_val.jsonl"

def parse_mcq_jsonl(filepath):
    records = []
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            data = json.loads(line)
            question = data.get('question', '')
            options = data.get('options', [])
            
            while len(options) < 4:
                options.append("")
            options = options[:4]
            
            records.append({
                'id': data.get('id'),
                'prompt': question,
                'A': options[0],
                'B': options[1],
                'C': options[2],
                'D': options[3],
                'label': data.get('answer_index', 0)
            })
    return pd.DataFrame(records)

df = parse_mcq_jsonl(MCQ_TRAIN_PATH)
df = df.dropna(subset=['label']).reset_index(drop=True)
df['label'] = df['label'].astype(int)

# Stratified Split
train_df, val_df = train_test_split(df, test_size=0.15, random_state=42, stratify=df['label'])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"✅ Training samples: {len(train_df)} | Validation samples: {len(val_df)}")

# ------------------------------------------------------------
# 2. TOKENIZATION (EXPANDED CONTEXT WINDOW)
# ------------------------------------------------------------
MODEL_NAME = "roberta-large"
MAX_LEN = 512  # Increased length window to retain full context

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
option_cols = ['A', 'B', 'C', 'D']

def preprocess_mcq(examples):
    # Construct explicit pairs: Question <sep> Option
    first_sentences = [[f"Question context: {prompt}"] * 4 for prompt in examples['prompt']]
    second_sentences = [[f"Option text: {examples[col][i]}" for col in option_cols] for i in range(len(examples['prompt']))]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized = tokenizer(
        first_sentences, 
        second_sentences, 
        truncation=True, 
        max_length=MAX_LEN
    )
    
    return {
        k: [v[i : i + 4] for i in range(0, len(v), 4)]
        for k, v in tokenized.items()
    }

@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0] else "labels"
        labels = [feature.pop(label_name) for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flattened = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)]
            for feature in features
        ]
        flattened = sum(flattened, [])

        batch = self.tokenizer.pad(
            flattened,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'macro_f1': f1, 'precision': precision, 'recall': recall}

train_ds = Dataset.from_pandas(train_df).map(preprocess_mcq, batched=True)
val_ds = Dataset.from_pandas(val_df).map(preprocess_mcq, batched=True)

# ------------------------------------------------------------
# 3. HIGH-PERFORMANCE REGULARIZED TRAINING
# ------------------------------------------------------------
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
output_dir = "./high_perf_mcq_model"

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    learning_rate=1e-5,               # Optimized learning rate
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    per_device_train_batch_size=2,    
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,    # Effective Batch Size = 16
    num_train_epochs=8,               # Extended training epochs for full convergence
    weight_decay=0.02,                # L2 Regularization prevents overfitting
    label_smoothing_factor=0.05,      # Label smoothing boosts generalization score
    max_grad_norm=1.0,                # Strictly controls loss explosion
    logging_steps=10,                 
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),   # FP16 enabled safely on RoBERTa
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("\n🚀 Executing High-Performance Training...")
trainer.train()

# Final Metrics
eval_res = trainer.evaluate()
print("\n================ VALIDATION METRICS ================")
print(f"Validation Loss     : {eval_res['eval_loss']:.4f}")
print(f"Validation Accuracy : {eval_res['eval_accuracy']:.4f}")
print(f"Validation Macro F1 : {eval_res['eval_macro_f1']:.4f}")
print("====================================================")

# ------------------------------------------------------------
# 4. SAVING BEST MODEL & METRICS
# ------------------------------------------------------------
SAVE_PATH = "./best_mcq_model"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

# Save validation metrics to output folder
with open(os.path.join(SAVE_PATH, "eval_results.json"), "w") as f:
    json.dump(eval_res, f, indent=4)

print(f"✅ Best Model and evaluation output saved under directory: '{SAVE_PATH}'")

# ------------------------------------------------------------
# 5. SUBMISSION FILE CREATION
# ------------------------------------------------------------
if os.path.exists(MCQ_VAL_PATH):
    test_df = parse_mcq_jsonl(MCQ_VAL_PATH)
    test_ds = Dataset.from_pandas(test_df).map(preprocess_mcq, batched=True)

    preds = trainer.predict(test_ds)
    final_test_preds = np.argmax(preds.predictions, axis=-1)

    submission_df = pd.DataFrame({
        'id': test_df['id'],
        'answer_index': final_test_preds
    })

    SUBMISSION_FILE = "sub_task2_fire2026.csv"
    submission_df.to_csv(SUBMISSION_FILE, index=False)
    print(f"🎉 SUBMISSION CSV SAVED AT: '{SUBMISSION_FILE}'")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
✅ Training samples: 850 | Validation samples: 150


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (5 outputs left out)


🚀 Executing High-Performance Training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Precision,Recall
1,11.298440,1.384812,0.260000,0.262219,0.273664,0.258824
2,11.151757,1.382064,0.326667,0.330274,0.334604,0.328636


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]